# 01 Foundation And Data Refresh

This notebook is the first step of the downstream Dutch 15-minute DAM extension.

Scope of the current notebook:

- lock the repo-native implementation contracts for the new quarter-hour extension;
- refresh the observed hourly and quarter-hour Dutch DAM inputs with the repository's own ingestion and cleaning scripts;
- keep the shared cleaned hourly and quarterly outputs separate;
- expose the dedicated quarterly continuation pipeline as a companion audit path instead of silently inventing a second framework.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

NOTEBOOK_CWD = Path.cwd()
REPO_ROOT = next(
    path
    for path in [NOTEBOOK_CWD, *NOTEBOOK_CWD.parents]
    if (path / "scripts/Data/02_Forecasting/01_DA_prices").exists()
)
os.chdir(REPO_ROOT)

PACKAGE_ROOT = REPO_ROOT / "scripts" / "Data" / "02_Forecasting" / "01_DA_prices"
if str(PACKAGE_ROOT) not in sys.path:
    sys.path.append(str(PACKAGE_ROOT))

from quarterhour_da import (
    QuarterHourDAExtensionConfig,
    assert_thesis_grade_actual_source_authorized,
    build_thesis_grade_frozen_actual_metadata,
    find_frozen_actual_version,
    find_latest_canonical_actual_run,
    find_latest_observed_deterministic_run,
    find_latest_phase01_run,
    find_latest_phase02_run,
    find_latest_phase03_run,
    find_latest_phase04_run,
    find_latest_phase07_run,
    find_latest_phase07_upstream_refresh_run,
    load_frozen_actual_diagnostics,
    load_frozen_actual_manifest,
    load_frozen_actual_path,
    resolve_frozen_actual_registry_entry,
    run_observed_market_deterministic_forecast,
)

config = QuarterHourDAExtensionConfig()
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)
plt.style.use("seaborn-v0_8-whitegrid")

## Optional Refresh Hook

The notebook defaults to loading the latest saved Phase 0/1 refresh artifact. Set `RUN_REFRESH = True` only when you want to rerun the update workflow from inside the notebook.

In [ ]:
RUN_REFRESH = False

if RUN_REFRESH:
    command = [
        sys.executable,
        str(REPO_ROOT / "scripts" / "Data" / "02_Forecasting" / "01_DA_prices" / "run_15min_phase01_refresh.py"),
    ]
    completed = subprocess.run(command, cwd=REPO_ROOT, capture_output=True, text=True, encoding="utf-8", errors="replace")
    print(completed.stdout)
    if completed.returncode != 0:
        print(completed.stderr)
        raise RuntimeError(f"Phase 0/1 refresh failed with exit code {completed.returncode}.")

In [ ]:
latest_run = find_latest_phase01_run(config)
if latest_run is None:
    raise FileNotFoundError("No saved Phase 0/1 refresh artifact exists yet. Run the phase refresh script first.")

foundation_contracts = pd.read_csv(latest_run / "foundation_contracts.csv")
refresh_summary = pd.read_csv(latest_run / "data_refresh_summary.csv")
command_status = pd.read_csv(latest_run / "command_status_summary.csv")
run_summary = json.loads((latest_run / "run_summary.json").read_text(encoding="utf-8"))

display(pd.DataFrame([{"latest_phase01_run": str(latest_run)}]))

## Repo-Native Foundation Contracts

These rows make the implementation choices explicit before any 15-minute modelling starts.

In [ ]:
display(foundation_contracts)

## Data Refresh Summary

This table compares the key hourly and quarter-hour cleaned datasets before and after the Phase 1 refresh.

In [ ]:
display(refresh_summary)

## Command Status

Each refresh step keeps its own log file so failures stay visible and auditable.

In [ ]:
display(command_status[["command_name", "status", "return_code", "log_path"]])